# カスケード型の音声対話システム

音声認識・LLM・音声合成をそれぞれ定義して試し、最後につないでマイクで会話します。3段階とも Gemini API を使います。
Google Colab で実行します。録音には、ブラウザのマイク使用許可が必要です。

In [ ]:
%pip install -q -U google-genai  # 音声認識・返答生成・音声合成に共通の SDK
%pip install -q ipywebrtc  # ブラウザのマイクで録音するためのライブラリ

## API の準備

[Google AI Studio](https://aistudio.google.com/apikey) で Gemini API キーを取得してください。
録音した音声と会話の内容は Gemini API へ送信します。利用枠・料金は [Google AI Studio](https://aistudio.google.com/) で確認できます。

In [ ]:
import os
from getpass import getpass
from google import genai

api_key = os.environ.get("GEMINI_API_KEY") or getpass("Gemini API キー: ")
client = genai.Client(api_key=api_key)
model_name = "gemini-flash-latest"  # 音声認識と返答生成
speech_model_name = "gemini-3.8-flash-tts"  # 音声合成

## 自分の声を文字にする

録音した音声データを受け取り、発話だけを書き起こすモジュールを定義します。

In [ ]:
from google.genai import types


def transcribe(audio_data: bytes) -> str:
    """録音した音声の書き起こし"""
    if not audio_data:
        raise ValueError("先にマイクで録音し、停止してから実行してください。")
    response = client.models.generate_content(
        model=model_name,
        contents=[
            "音声の発話だけを、話された言語のまま書き起こしてください。"
            "質問には答えず、書き起こした文だけを返してください。"
            "発話がなければ空文字を返してください。",
            types.Part.from_bytes(data=audio_data, mime_type="audio/webm"),
        ],
        config=types.GenerateContentConfig(temperature=0),
    )
    return (response.text or "").strip()

### マイク録音の準備

録音の開始・停止と、Python への音声データの受け渡しを行うウィジェットを用意します。Google Chrome での利用を想定しています。

In [ ]:
from google.colab import output
from ipywebrtc import AudioRecorder, CameraStream
from ipywidgets import Button
from IPython.display import display

output.enable_custom_widget_manager()


def create_recorder():
    """権限取得と録音ウィジェットの準備"""
    mic = CameraStream(constraints={"audio": True, "video": False})
    mic.layout.display = "none"  # 権限を取得し、カメラ用のライブ表示は隠す
    recorder = AudioRecorder(stream=mic, format="webm")
    finish = Button(description="終了")

    def release_mic(change):
        if change["new"]:
            recorder.stream.close()
            recorder.stream = CameraStream(constraints={"audio": True, "video": False})

    def close_recorder(_):
        recorder.recording = False
        recorder.send_state("recording")

    recorder.audio.observe(release_mic, names="value")
    finish.on_click(close_recorder)
    display(mic, recorder, finish)
    return recorder

### 実演

次のセルで録音ウィジェットを表示し、録音ボタンを押してください。初回はブラウザのマイク使用を許可します。
「日本の首都はどこですか？」などと話して録音を停止し、その下のセルで書き起こします。自分の発話と合っているか確認しましょう。

In [ ]:
recorder = create_recorder()

In [ ]:
audio_data = bytes(recorder.audio.value)
transcript = transcribe(audio_data)
print("書き起こし:", transcript)

## LLM で返答を作る

会話履歴を保持する LLM を準備します。まずはテキストの質問を送り、返答を確かめます。

In [ ]:
from google.genai import types


def create_chat():
    """会話履歴を保持する LLM の準備"""
    return client.chats.create(
        model=model_name,
        config=types.GenerateContentConfig(
            system_instruction="フレンドリーかつ自然な日本語で、1〜2文で簡潔に返答してください。箇条書きや Markdown は使わないでください。",
        ),
    )

In [ ]:
chat = create_chat()
question = "日本の首都はどこですか？"
text_reply = chat.send_message(question).text or ""
print("質問:", question)
print("返答:", text_reply)

## 文字を音声に変える

文章を読み上げるモジュールを定義します。返された WAV データを、そのまま音声プレーヤーへ渡します。

In [ ]:
from google.genai import types


def synthesize(text: str) -> bytes:
    """文章の音声合成と WAV データの取得"""
    if not text.strip():
        raise ValueError("読み上げる文章が空です。")
    response = client.models.generate_content(
        model=speech_model_name,
        contents=text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(voice="Kore"),
            ),
        ),
    )
    audio = next(
        part.inline_data
        for part in response.candidates[0].content.parts
        if part.inline_data is not None
    )
    return audio.data

In [ ]:
from IPython.display import Audio, display

reply_audio = synthesize(text_reply)
display(Audio(reply_audio))

## マイクで会話する

次のセルを実行し、録音ウィジェットの録音ボタンを押して話してください。
録音を停止すると、書き起こし・返答生成・音声合成が順に進みます。返答を聴き終えたら、再び録音して会話を続けられます。「終了」ボタンでも録音を停止できます。
自動再生されない場合は、返答の再生ボタンを押してください。セルを再実行すると新しい会話になります。

参考：[Gemini の音声入力](https://ai.google.dev/gemini-api/docs/audio)、[音声合成](https://ai.google.dev/gemini-api/docs/speech-generation)。

In [ ]:
from ipywidgets import Output

chat = create_chat()
dialogue_output = Output()


@dialogue_output.capture(clear_output=True, wait=True)
def respond(change):
    """録音した音声への応答"""
    if not change["new"]:
        return
    transcript = transcribe(bytes(change["new"]))
    if not transcript:
        print("発話を聞き取れませんでした。もう一度録音してください。")
        return
    print("あなた:", transcript)
    text_reply = chat.send_message(transcript).text or ""
    print("Gemini:", text_reply)
    display(Audio(synthesize(text_reply), autoplay=True))


dialogue_recorder = create_recorder()
dialogue_recorder.audio.observe(respond, names="value")  # 録音データが届いたら応答
display(dialogue_output)

## まとめ

音声認識 → 返答生成 → 音声合成を順に行う、単純なカスケード構成です。
処理を順番に待つため、リアルタイムの会話には程遠く、遅延を減らすさまざまな工夫が必要です。

より先端的なシステムとして、ユーザーとシステムの音声を同時に扱い、聞きながら話す双方向対話型のモデルもあります（例：[J-Moshi](https://github.com/nu-dialogue/j-moshi)）。